# End-to-end pipeline composition test (Colab + Drive)

Standalone Colab notebook that runs the full deployment chain on a single audio file. **No repo clone, no `modules/`, no FAISS, no Mistral.** Every model is loaded inline directly from HF or from your Drive-resident v4 fine-tune.

```
audio.wav
  ──► M1  Whisper-yo v4 (from Drive)         → YO text (raw)
        ──► M2  Davlan/mT5_base_yoruba_adr   → YO text (diacritized)
              ──► M3  NLLB-200 yor → eng     → EN query
                    ──► M4  Qwen2.5-1.5B     → EN answer
                          ──► M5a NLLB eng → yor  → YO answer text
                                ──► M5b mms-tts-yor → response.wav
```

**What you need**:
- Colab runtime with a GPU (T4 fine; A100 obviously faster).
- A finished `Whisper_v4.ipynb` training run on Drive with `merged_16bit/` saved.
- HF_TOKEN in Colab Secrets (only used for HF auth on private repos; not strictly required for these public models).

Step-by-step cells. Each model loads once and stays in VRAM; the final chain reuses everything.

## Step 1 — Auth + Drive mount

Only prompt cell. Authorize Drive, walk away.

In [ ]:
import os
from pathlib import Path

from google.colab import drive
try:
    drive.mount("/content/drive", force_remount=False)
except ValueError:
    drive.mount("/content/drive", force_remount=True)

DRIVE_TRAINING = Path("/content/drive/MyDrive/yoruba-pipeline-logs/training")
print(f"Drive mounted → {DRIVE_TRAINING}")

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("HF auth: ok")
else:
    print("HF auth: skipped — public models still work.")

## Step 2 — Install dependencies

In [ ]:
%%capture
!pip install -q "transformers>=4.45" "datasets>=3.0" "huggingface_hub>=0.24" \
    librosa soundfile accelerate sentencepiece protobuf

## Step 3 — Config

`V4_PATH = None` → auto-discover the most recent v4 training run on Drive. Override to pin a specific run.

`INPUT_AUDIO` defaults to a small Yorùbá FLEURS sample (downloaded on first run). Replace with any 16 kHz mono `.wav` or `.mp3` to test other input.

In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32

# Model identifiers (all public)
BASELINE_M1_ID = "openai/whisper-large-v3"
M2_MODEL_ID    = "Davlan/mT5_base_yoruba_adr"
NLLB_MODEL_ID  = "facebook/nllb-200-distilled-600M"
M4_MODEL_ID    = "Qwen/Qwen2.5-1.5B-Instruct"
M5_TTS_ID      = "facebook/mms-tts-yor"

# v4 fine-tune from Drive (None = auto-pick latest with merged_16bit/)
V4_PATH = None
if V4_PATH is None:
    runs = sorted(
        [p for p in DRIVE_TRAINING.iterdir() if (p / "merged_16bit").exists()],
        reverse=True,
    ) if DRIVE_TRAINING.exists() else []
    if not runs:
        raise FileNotFoundError(
            f"No run with merged_16bit/ under {DRIVE_TRAINING}. "
            "Run Whisper_v4.ipynb to completion first."
        )
    V4_PATH = str(runs[0] / "merged_16bit")
    V4_RUN  = runs[0].name
else:
    V4_RUN = Path(V4_PATH).parent.name

# Input audio (defaults to a Yorùbá FLEURS sample)
INPUT_AUDIO  = Path("/content/fleurs_yo_sample.wav")
OUTPUT_AUDIO = Path("/content/response.wav")

if not INPUT_AUDIO.exists():
    print("Fetching a Yorùbá FLEURS sample (one-time)…")
    from datasets import load_dataset, Audio
    import soundfile as sf
    fleurs = load_dataset("google/fleurs", "yo_ng", split="test", streaming=True)
    sample = next(iter(fleurs.cast_column("audio", Audio(sampling_rate=16000))))
    sf.write(str(INPUT_AUDIO), sample["audio"]["array"], 16000)
    print(f"  wrote {INPUT_AUDIO}")

print(f"\ndevice : {DEVICE} ({DTYPE})")
print(f"V4 run : {V4_RUN}")
print(f"V4 path: {V4_PATH}")
print(f"input  : {INPUT_AUDIO}")

## M1 — ASR (audio → raw Yorùbá)

Loads the v4 Whisper fine-tune directly from Drive. First load streams ~3 GB from Drive (slow once, ~2–4 min); after that it's cached on the runtime.

Flip `M1_USE_BASELINE = True` to load `openai/whisper-large-v3` instead for an A/B compare.

In [ ]:
import librosa
import soundfile as sf
from IPython.display import Audio, display
from transformers import WhisperProcessor, WhisperForConditionalGeneration

M1_USE_BASELINE = False  # True → openai/whisper-large-v3, False → v4 from Drive
M1_MODEL = BASELINE_M1_ID if M1_USE_BASELINE else V4_PATH

print(f"loading M1 from {M1_MODEL}…")
m1_proc  = WhisperProcessor.from_pretrained(M1_MODEL)
m1_model = WhisperForConditionalGeneration.from_pretrained(
    M1_MODEL, torch_dtype=DTYPE,
).to(DEVICE).eval()
m1_model.generation_config.language = "<|yo|>"
m1_model.generation_config.task = "transcribe"
m1_model.generation_config.forced_decoder_ids = None

# Load input audio at 16 kHz mono
audio, sr = librosa.load(str(INPUT_AUDIO), sr=16000, mono=True)

# Transcribe
feats = m1_proc.feature_extractor(
    audio, sampling_rate=16000, return_tensors="pt",
).input_features.to(DEVICE, dtype=DTYPE)
with torch.inference_mode():
    ids = m1_model.generate(
        feats, language="<|yo|>", task="transcribe",
        max_new_tokens=256, num_beams=1,
    )
yo_raw = m1_proc.tokenizer.batch_decode(ids, skip_special_tokens=True)[0].strip()

print(f"\nYO (raw): {yo_raw}")
display(Audio(str(INPUT_AUDIO)))

## M2 — Diacritic restoration (raw YO → diacritized YO)

`Davlan/mT5_base_yoruba_adr` restores tone marks and sub-dotted letters. The v4 fine-tune already outputs diacritized text most of the time, so M2 is mostly a safety net — kept in the chain for robustness.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print(f"loading M2 from {M2_MODEL_ID}…")
m2_tok = AutoTokenizer.from_pretrained(M2_MODEL_ID)
m2_mdl = AutoModelForSeq2SeqLM.from_pretrained(M2_MODEL_ID).to(DEVICE).eval()

inputs = m2_tok(yo_raw, return_tensors="pt", truncation=True).to(DEVICE)
with torch.inference_mode():
    out = m2_mdl.generate(**inputs, max_new_tokens=256, num_beams=4)
yo_diacritized = m2_tok.decode(out[0], skip_special_tokens=True).strip()

print(f"\nYO (diacrit.): {yo_diacritized}")

## M3 — Translation YO → EN

NLLB-200 distilled-600M, `yor_Latn` → `eng_Latn`. Loaded once and reused in M5a (en → yo) with a different `src_lang` / `forced_bos_token_id`.

In [ ]:
print(f"loading NLLB from {NLLB_MODEL_ID}…")
nllb_mdl = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL_ID).to(DEVICE).eval()

# YO → EN
m3_tok = AutoTokenizer.from_pretrained(NLLB_MODEL_ID, src_lang="yor_Latn")

inputs = m3_tok(yo_diacritized, return_tensors="pt", truncation=True).to(DEVICE)
with torch.inference_mode():
    out = nllb_mdl.generate(
        **inputs,
        forced_bos_token_id=m3_tok.convert_tokens_to_ids("eng_Latn"),
        max_new_tokens=256, num_beams=4,
    )
en_query = m3_tok.batch_decode(out, skip_special_tokens=True)[0].strip()

print(f"\nEN query: {en_query}")

## M4 — Small LLM (EN query → EN answer)

Qwen2.5-1.5B-Instruct. Fits comfortably alongside everything else in VRAM. Direct factual answers — no retrieval. Greedy decoding (`do_sample=False`) for deterministic output.

In [ ]:
from transformers import AutoModelForCausalLM

print(f"loading M4 from {M4_MODEL_ID}…")
m4_tok = AutoTokenizer.from_pretrained(M4_MODEL_ID)
m4_mdl = AutoModelForCausalLM.from_pretrained(
    M4_MODEL_ID, torch_dtype="auto", device_map="auto",
).eval()

messages = [
    {"role": "system",
     "content": "You are a concise assistant. Answer in 2-3 short sentences."},
    {"role": "user", "content": en_query},
]
prompt = m4_tok.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt",
).to(m4_mdl.device)
with torch.inference_mode():
    out = m4_mdl.generate(
        prompt, max_new_tokens=200, do_sample=False,
        pad_token_id=m4_tok.eos_token_id,
    )
en_answer = m4_tok.decode(out[0, prompt.shape[-1]:], skip_special_tokens=True).strip()

print(f"\nEN answer: {en_answer}")

## M5a — Translation EN → YO

Reuses the NLLB model loaded in M3 with a new tokenizer configured for `eng_Latn` → `yor_Latn`.

In [ ]:
m5a_tok = AutoTokenizer.from_pretrained(NLLB_MODEL_ID, src_lang="eng_Latn")

inputs = m5a_tok(en_answer, return_tensors="pt", truncation=True).to(DEVICE)
with torch.inference_mode():
    out = nllb_mdl.generate(
        **inputs,
        forced_bos_token_id=m5a_tok.convert_tokens_to_ids("yor_Latn"),
        max_new_tokens=256, num_beams=4,
    )
yo_answer = m5a_tok.batch_decode(out, skip_special_tokens=True)[0].strip()

print(f"YO answer (text): {yo_answer}")

## M5b — TTS (YO text → audio)

`facebook/mms-tts-yor` (VITS-based). Writes to `OUTPUT_AUDIO` and plays inline.

In [ ]:
from transformers import VitsModel

print(f"loading M5b TTS from {M5_TTS_ID}…")
tts_tok = AutoTokenizer.from_pretrained(M5_TTS_ID)
tts_mdl = VitsModel.from_pretrained(M5_TTS_ID).to(DEVICE).eval()

inputs = tts_tok(yo_answer, return_tensors="pt").to(DEVICE)
with torch.inference_mode():
    waveform = tts_mdl(**inputs).waveform.cpu().squeeze().numpy()

sr_out = tts_mdl.config.sampling_rate
sf.write(str(OUTPUT_AUDIO), waveform, sr_out)

print(f"\nwrote {OUTPUT_AUDIO} ({len(waveform)/sr_out:.2f}s @ {sr_out} Hz)")
display(Audio(str(OUTPUT_AUDIO)))

## Full chain summary

Prints every stage's input/output side by side. Useful for inspecting where the chain succeeds or fails on a given input.

In [ ]:
print("=" * 70)
print(f"INPUT AUDIO   : {INPUT_AUDIO}")
print("=" * 70)
print(f"M1  YO raw    : {yo_raw}")
print(f"M2  YO diacrit: {yo_diacritized}")
print(f"M3  EN query  : {en_query}")
print(f"M4  EN answer : {en_answer}")
print(f"M5a YO answer : {yo_answer}")
print(f"M5b OUTPUT    : {OUTPUT_AUDIO}")
print("=" * 70)

print("\nInput:")
display(Audio(str(INPUT_AUDIO)))
print("Output:")
display(Audio(str(OUTPUT_AUDIO)))

## Re-run on a new audio file

All models stay loaded. Change `INPUT_AUDIO` to a new path and re-run cells `m1` → `summary` (no need to re-load any model).

In [ ]:
# Example: upload your own audio in Colab
RUN_UPLOAD = False

if RUN_UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    # Take the first uploaded file as the new input
    fname = next(iter(uploaded))
    INPUT_AUDIO = Path("/content") / fname
    print(f"INPUT_AUDIO now: {INPUT_AUDIO}")
    print("Re-run cells m1 → summary to transcribe + chain.")
else:
    print("RUN_UPLOAD=False — flip to True and re-run to upload audio.")